## 9. Re-evaluation with a 4096-token output budget (Phase 7, step 1)

Add these cells **at the end** of `vigraph_qlora_windows.ipynb`. Before running them, run
**section 1** (settings + helpers). Training is not repeated, and the dataset must already exist
(section 2).

**Why:** the first evaluation used `--max-new-tokens 2048`. On the hardest diagrams (L4), the
answer was cut off for zero-shot on 27 of 28 samples and for the fine-tuned model on 11 of 28, so
L4 measured the output budget more than the model. These runs repeat the same 112 test samples,
image size and greedy decoding with **4096** tokens (the app's default).

- New run names end in `-tok4096`, so the first results stay untouched.
- Each run resumes after a stop: just run its cell again.
- The OCR baseline has no token budget, so its existing run is reused for the comparisons.
- Expect roughly 2.5–4 h per model (L4 answers are now allowed to be long).

In [ ]:
# 9.1 Settings for the re-evaluation (uses section 1's helpers)
import yaml

MAX_NEW_TOKENS_V2 = 4096
EXPECTED = EVAL_LIMIT or 500
model_cfg = yaml.safe_load(CONFIG.read_text(encoding="utf-8"))["model"]
model_slug = model_cfg["id"].split("/")[-1].lower()
hf_v2 = ["--backend", "hf", "--model-id", model_cfg["id"], "--dtype", COMPUTE_DTYPE,
         "--device-map", "cuda:0", "--max-new-tokens", MAX_NEW_TOKENS_V2]
if model_cfg.get("revision"):
    hf_v2 += ["--revision", model_cfg["revision"]]

NAMES_V2 = {
    "qlora": f"{RUN_NAME}-tok4096-s0",
    "zeroshot": f"{model_slug}-zeroshot-tok4096-s0",
}
QLORA_V2 = EVAL_DIR / NAMES_V2["qlora"]
ZEROSHOT_V2 = EVAL_DIR / NAMES_V2["zeroshot"]
BASELINE_RUN = EVAL_DIR / "baseline-v1-windows"      # from section 7 (no token budget)
QLORA_2048 = EVAL_DIR / f"{RUN_NAME}-s0"              # the first evaluation, for the budget effect


def evaluate_v2(name, *args):
    """Run (or resume) one evaluation on the same test samples; skip it when finished."""
    out = EVAL_DIR / name
    if (out / "summary.json").exists() and (out / "run.json").exists():
        info = read_json(out / "run.json")
        if info.get("completed_at") and info.get("samples_done") == EXPECTED:
            print(f"{name}: done")
            return out
    command = ["module", "evaluation", "run", "--dataset", DATA_DIR, "--split", "test",
               "--out", out, "--image-max-side", image_side(), *args]
    if EVAL_LIMIT:
        command += ["--limit", EVAL_LIMIT]
    driver(*command, log=f"eval-{name}")
    errors = read_json(out / "run.json").get("errors", 0)
    if errors:
        print(f"\nNote: {errors} sample(s) raised an error in {name}; they count as failures.")
    return out


if not (RUN_DIR / "adapter" / "adapter_config.json").exists():
    raise RuntimeError(f"No trained adapter in {RUN_DIR / 'adapter'} — section 6 must have finished.")
print(f"image_max_side {image_side()} | max_new_tokens {MAX_NEW_TOKENS_V2} | "
      f"{EXPECTED} test samples | runs → {EVAL_DIR}")

In [ ]:
# 9.2 Fine-tuned model, 4096 tokens (the most important run — resumable)
qlora_v2 = evaluate_v2(NAMES_V2["qlora"], *hf_v2, "--adapter", RUN_DIR / "adapter")

In [ ]:
# 9.3 Zero-shot model, 4096 tokens (resumable)
zeroshot_v2 = evaluate_v2(NAMES_V2["zeroshot"], *hf_v2)

In [ ]:
# 9.4 Paired comparisons (§20.11) on the same 112 samples
comparisons_v2 = EVAL_DIR / "comparisons-tok4096"
comparisons_v2.mkdir(parents=True, exist_ok=True)
pairs_v2 = {
    "qlora-vs-zeroshot": (ZEROSHOT_V2, QLORA_V2),
    "qlora-vs-baseline": (BASELINE_RUN, QLORA_V2),
    "zeroshot-vs-baseline": (BASELINE_RUN, ZEROSHOT_V2),
    "qlora4096-vs-qlora2048": (QLORA_2048, QLORA_V2),   # the effect of the token budget alone
}
for name, (a, b) in pairs_v2.items():
    missing = [p.name for p in (a, b) if not (p / "summary.json").exists()]
    if missing:
        print(f"skipped {name}: no finished results for {', '.join(missing)}")
        continue
    driver("module", "evaluation", "compare", "--a", a, "--b", b,
           "--out", comparisons_v2 / f"{name}.md", log="compare-tok4096")

In [ ]:
# 9.5 Headline table and truncation by level (reads the run files only)
import gzip

keys = ("node_f1", "edge_f1", "edge_strict_f1", "graph_similarity", "label_accuracy",
        "qa_accuracy", "valid_first_attempt", "valid_post_repair")
header = ("run", "node F1", "edge F1", "strict", "graph sim", "labels", "QA", "valid@1", "valid")
table = {"baseline": BASELINE_RUN, "zeroshot-2048": EVAL_DIR / f"{model_slug}-zeroshot-s0",
         "qlora-2048": QLORA_2048, "zeroshot-4096": ZEROSHOT_V2, "qlora-4096": QLORA_V2}
print(" | ".join(header))
for name, path in table.items():
    if not (path / "summary.json").exists():
        continue
    macro = read_json(path / "summary.json")["macro"]
    values = [macro.get(k, {}).get("mean") for k in keys]
    print(" | ".join([name] + ["–" if v is None else f"{v:.3f}" for v in values]))


def truncation_by_level(path):
    counts = {}
    raw = path / "predictions.jsonl"
    text = raw.read_text(encoding="utf-8") if raw.exists() else gzip.decompress(
        (path / "predictions.jsonl.gz").read_bytes()).decode("utf-8")
    for line in text.splitlines():
        if not line.strip():
            continue
        result = json.loads(line)
        level = result["attributes"]["level"]
        analysis = result["prediction"].get("analysis") or {}
        attempts = analysis.get("extraction", {}).get("attempts", [])
        cut = any(a["output"].get("finish_reason") == "length" for a in attempts)
        failed = not result["scores"]["has_graph"]
        n, t, f = counts.get(level, (0, 0, 0))
        counts[level] = (n + 1, t + cut, f + failed)
    return counts


print("\ntruncated / failed per level (of n):")
for name in ("zeroshot-2048", "zeroshot-4096", "qlora-2048", "qlora-4096"):
    path = table[name]
    if (path / "run.json").exists():
        counts = truncation_by_level(path)
        print(f"  {name:14s}", "  ".join(f"L{lv}: {t}/{f} of {n}" for lv, (n, t, f) in sorted(counts.items())))

In [ ]:
# 9.6 Hand-back: one small zip (no adapter — only the new evaluations and comparisons)
handback_v2 = OUT / "handback" / f"vigraph-handback-{RUN_NAME}-tok4096.zip"
extra_v2 = sorted((EVAL_DIR / "comparisons-tok4096").glob("*"))
finished = [p for p in (QLORA_V2, ZEROSHOT_V2) if (p / "summary.json").exists()]
driver("handback", "--out", handback_v2, "--eval", *finished, "--extra", *extra_v2, log="handback")
print("\nGive back:", handback_v2)

### Files to give back

**`F:\vigraph\output\handback\vigraph-handback-qwen3vl-2b-qlora-a6000-v1-tok4096.zip`**: a
few hundred KB, small enough to upload directly. It contains the two new evaluation runs
(`run.json`, `summary.json`, `report.md`, `predictions.jsonl.gz`), the four comparisons, and
`MANIFEST.json`. Also paste the output of cell 9.5.